# Feature Engineering Pipeline Demo

`analytics_toolbox.feature_engineering` — windowed aggregate features with leakage safety.

This notebook walks through the full pipeline end-to-end:
1. Create a fixture (synthetic Medicaid claims data)
2. Build an entity-date spine
3. Compute windowed aggregate features (pharmacy, medical utilization)
4. Expand grains via group columns
5. Join multiple feature sets
6. Protect against fan-out with guardrails

**Privacy guarantee:** Temporal boundaries are strictly enforced — no forward leakage, exclusive upper bound on as-of date.

## 0. Setup

Install the package (skip if already installed):
```bash
pip install analytics-toolbox
```

In [1]:

import duckdb
import pandas as pd

from analytics_toolbox.feature_engineering import (
    Agg,
    Guardrails,
    compute_features,
    join_features,
)
from analytics_toolbox.feature_engineering.examples.medicaid_features import (
    med_utilization_features,
    rx_spend_features,
)
from analytics_toolbox.feature_engineering.fixtures.medicaid import (
    make_medicaid_fixture,
    make_spine,
)

print("Imports successful.")

Imports successful.


## 1. Create a Medicaid Fixture

Synthetic but realistic claims data: members, pharmacy (RX), and medical (ED, inpatient) claims.

In [2]:
con = duckdb.connect()
make_medicaid_fixture(con, n_members=500, seed=42)

members = con.execute("SELECT * FROM members LIMIT 5").df()
print("Members table:")
print(members)

rx_claims = con.execute("SELECT * FROM rx_claims LIMIT 5").df()
print("\nRx claims table:")
print(rx_claims)

Members table:
   member_id        dob sex eligibility_category county_fips enroll_start  \
0          1 1997-05-13   M                 aged       06037   2023-01-01   
1          2 1961-12-20   M       blind_disabled       01001   2023-01-01   
2          3 2000-09-16   M             pregnant       17031   2023-01-01   
3          4 1942-11-07   M                 aged       01003   2023-01-01   
4          5 1960-11-14   M             pregnant       01003   2023-01-01   

  enroll_end  
0 2025-12-31  
1 2025-12-31  
2 2025-12-31  
3 2025-12-31  
4 2025-12-31  

Rx claims table:
   claim_id  member_id claim_date  ndc_code      drug_class  paid_amount  \
0         2          2 2024-06-30  NDC58567     psychiatric       283.02   
1         3          3 2024-05-31  NDC71141       analgesic       137.40   
2         4          3 2024-05-30  NDC22689       analgesic       148.65   
3         5          4 2024-03-04  NDC42541      antibiotic       407.67   
4         6          4 2023-07-14 

## 2. Create an Entity-Date Spine

A spine is the grain of your model — one row per (member, as_of_date) combination.

In [3]:
as_of_dates = pd.date_range('2024-01-01', periods=10, freq='ME').tolist()
member_ids = list(range(1001, 1051))
spine = make_spine(con, as_of_dates, member_ids=member_ids)

print(f"Spine shape: {spine.shape}")
print(f"Columns: {list(spine.columns)}")
print(f"Date range: {spine['as_of_date'].min()} to {spine['as_of_date'].max()}")

Spine shape: (500, 2)
Columns: ['member_id', 'as_of_date']
Date range: 2024-01-31 00:00:00 to 2024-10-31 00:00:00


## 3. Compute Windowed Features — Pharmacy Spend

Aggregate pharmacy claims within 30, 90, and 365-day lookback windows.

In [4]:
rx_features = rx_spend_features(spine, con, windows=(30, 90, 365))

print(f"RX features shape: {rx_features.shape}")
print(f"Columns: {list(rx_features.columns)}")
print("\nFirst 5 rows:")
print(rx_features.head())

RX features shape: (500, 14)
Columns: ['member_id', 'as_of_date', 'rx__paid_sum_30d', 'rx__claims_cnt_30d', 'rx__ndc_ndist_30d', 'rx__dsupply_sum_30d', 'rx__paid_sum_90d', 'rx__claims_cnt_90d', 'rx__ndc_ndist_90d', 'rx__dsupply_sum_90d', 'rx__paid_sum_365d', 'rx__claims_cnt_365d', 'rx__ndc_ndist_365d', 'rx__dsupply_sum_365d']

First 5 rows:
   member_id as_of_date  rx__paid_sum_30d  rx__claims_cnt_30d  \
0       1001 2024-01-31               NaN                   0   
1       1002 2024-01-31               NaN                   0   
2       1003 2024-01-31               NaN                   0   
3       1004 2024-01-31               NaN                   0   
4       1005 2024-01-31               NaN                   0   

   rx__ndc_ndist_30d  rx__dsupply_sum_30d  rx__paid_sum_90d  \
0                  0                  NaN               NaN   
1                  0                  NaN               NaN   
2                  0                  NaN            160.17   
3             

## 4. Medical Utilization Features

Conditional aggregates (ED visits, inpatient stays).

In [5]:
med_features = med_utilization_features(spine, con, windows=(30, 90, 365))

print(f"Med features shape: {med_features.shape}")
print("\nFirst 5 rows:")
print(med_features.head())

Med features shape: (500, 17)

First 5 rows:
   member_id as_of_date  med__claims_cnt_30d  med__paid_sum_30d  \
0       1001 2024-01-31                    0                NaN   
1       1002 2024-01-31                    0                NaN   
2       1003 2024-01-31                    0                NaN   
3       1006 2024-01-31                    0                NaN   
4       1007 2024-01-31                    1            4594.71   

   med__ed_visits_cnt_30d  med__inpatient_cnt_30d  med__provider_ndist_30d  \
0                     NaN                     NaN                        0   
1                     NaN                     NaN                        0   
2                     NaN                     NaN                        0   
3                     NaN                     NaN                        0   
4                     0.0                     0.0                        1   

   med__claims_cnt_90d  med__paid_sum_90d  med__ed_visits_cnt_90d  \
0             

## 5. Join Multiple Feature Sets

Combine pharmacy and medical features into a single table.

In [6]:
combined = join_features([rx_features, med_features], on=['member_id', 'as_of_date'])

print(f"Combined features shape: {combined.shape}")
feature_cols = [c for c in combined.columns if c not in ['member_id', 'as_of_date']]
print(f"Total feature columns: {len(feature_cols)}")

Combined features shape: (500, 29)
Total feature columns: 27


## 6. Custom Aggregations

Write any SQL aggregate expression — percentiles, min/max, conditionals.

In [7]:
custom_aggs = [
    Agg('paid_mean', 'AVG(paid_amount)'),
    Agg('paid_max', 'MAX(paid_amount)'),
    Agg('paid_min', 'MIN(paid_amount)'),
    Agg('high_cost_cnt', 'SUM(CASE WHEN paid_amount > 100 THEN 1 ELSE 0 END)'),
]
custom_features = compute_features(
    spine.head(50),
    'rx_claims',
    entity_keys=['member_id'],
    as_of_col='as_of_date',
    base_date_col='claim_date',
    namespace='custom',
    aggregations=custom_aggs,
    windows=[30, 90],
    con=con,
)
print(f'Custom features shape: {custom_features.shape}')
print(f'Columns: {list(custom_features.columns)}')

Custom features shape: (50, 10)
Columns: ['member_id', 'as_of_date', 'custom__paid_mean_30d', 'custom__paid_max_30d', 'custom__paid_min_30d', 'custom__high_cost_cnt_30d', 'custom__paid_mean_90d', 'custom__paid_max_90d', 'custom__paid_min_90d', 'custom__high_cost_cnt_90d']


## 7. Guardrails — Fan-Out Protection

Estimate pre-aggregation row count and warn/raise if it exceeds threshold.

In [8]:
small_as_of = as_of_dates[:5]
small_members = [1001, 1002, 1003, 1004, 1005]
small_spine = make_spine(con, small_as_of, member_ids=small_members)

result = compute_features(
    small_spine,
    'rx_claims',
    entity_keys=['member_id'],
    as_of_col='as_of_date',
    base_date_col='claim_date',
    namespace='rxtest',
    aggregations=[Agg('paid_sum', 'SUM(paid_amount)')],
    windows=[30],
    con=con,
    guardrails=Guardrails(max_fanout_rows=50_000_000),
)
print(f'Small spine computed: {result.shape}')

Small spine computed: (25, 3)


## 8. Stress Test — Larger Spine

Performance on realistic scale: 1000 members, 12 months.

In [9]:
import time

con_large = duckdb.connect()
make_medicaid_fixture(con_large, n_members=1000, seed=99)

large_as_of_dates = pd.date_range('2024-01-01', periods=12, freq='ME').tolist()
large_member_ids = list(range(1001, 2001))
large_spine = make_spine(con_large, large_as_of_dates, member_ids=large_member_ids)

print(f'Large spine: {large_spine.shape}')

start = time.time()
large_rx = rx_spend_features(large_spine, con_large, windows=(30, 90, 365))
rx_elapsed = time.time() - start

start = time.time()
large_med = med_utilization_features(large_spine, con_large, windows=(30, 90, 365))
med_elapsed = time.time() - start

print(f'RX features: {rx_elapsed:.2f}s')
print(f'Med features: {med_elapsed:.2f}s')
print(f'Total: {rx_elapsed + med_elapsed:.2f}s')

Large spine: (12000, 2)
RX features: 0.02s
Med features: 0.02s
Total: 0.04s
